# Stuttering Detection: Neural Network (Deep Learning) Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Focus**: Shallow Architecture vs. Deep Multi-Layer Perceptrons

---

## Step 1: Initialization

In [1]:
import os
import shutil
import numpy as np
import torch.nn as nn
import torch.optim as optim
from src.extractors import WavLMExtractor
from src.data import DataManager
from src.models import ShallowNeuralNetwork, DeepNeuralNetwork

# Dataset Configuration
AUDIO_DIR = "Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

/home/anshuman139/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2 (Optional): Operational Mode for Data Extraction
* `SKIP_EXTRACTION`: Uses features already on disk (Default).
* `FORCE_EXTRACT`: Analyzes raw audio for new files (Resumable).
* `CLEAN_START`: Wipes the database and re-extracts from zero.

In [ ]:
# Operational Flags
SKIP_EXTRACTION = True
FORCE_EXTRACT = False
CLEAN_START = False
NUM_CLIPS_TO_EXTRACT = 1000

if CLEAN_START:
    if os.path.exists(FEATURE_DIR):
        shutil.rmtree(FEATURE_DIR)
    print("[System] Clean start initiated. Feature database wiped.")

if not SKIP_EXTRACTION or CLEAN_START or FORCE_EXTRACT:
    extractor = WavLMExtractor("microsoft/wavlm-base")
    label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
    
    # Native randomized sampling of the dataset
    extractor.extract_from_dir(
        AUDIO_DIR, 
        output_dir=FEATURE_DIR, 
        label_dict=label_dict, 
        limit=NUM_CLIPS_TO_EXTRACT, 
        random_sample=True
    )
else:
    print("[System] Skipping extraction. Using existing data on disk.")

## Step 3: Data Preparation

In [2]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
manager = DataManager(None, None)

# Loading .npy features
X, y = manager.load_from_folders(fluent_dir, disfluent_dir)
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)

# Balance Training Set (50/50 split)
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

# Final Scaling
X_train_final = manager.preprocess(X_train_bal, method="standard")
X_test_final = manager.preprocess(X_test, method="standard")

[DataManager] Quality Filter: Removed 3938 low-quality samples.


## Step 4: Model 1 - Shallow Neural Network
Our shallow architecture uses a single hidden layer of 64 neurons. 

**Explicit Configuration**: 
* `activation_fn`: Sigmoid (Classical probabilistic non-linearity)
* `optimizer_class`: SGD (Stochastic Gradient Descent)
* `lr`: Learning Rate (Controls convergence speed)

In [3]:
# --- [Hyperparameter Panel] ---
# Activation: nn.Sigmoid, nn.ReLU, nn.Tanh, nn.LeakyReLU
# Optimizer:  optim.SGD, optim.Adam, optim.Adagrad
# LR values:   0.01 (Standard), 0.001 (Slow/Stable), 0.1 (Aggressive)

shallow_nn = ShallowNeuralNetwork(
    model_name="Shallow_NN_64", 
    hidden_layer_size=64, 
    epochs=200,
    lr=0.01,
    activation_fn=nn.Sigmoid,
    optimizer_class=optim.SGD
)

shallow_nn.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
shallow_nn.evaluate(X_test_final, y_test)

[Model: Shallow_NN_64] Initialized.
[Shallow_NN_64] Training PyTorch Network (Layers: [64]) for 200 epochs...


/home/anshuman139/venv/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


   Epoch [50/200], Loss: 0.6863
   Epoch [100/200], Loss: 0.6798
   Epoch [150/200], Loss: 0.6738
   Epoch [200/200], Loss: 0.6678

--- Evaluation on Unseen Test Set ---

--- Evaluation: Shallow_NN_64 ---
Accuracy: 0.6194
Precision: 0.4765
Recall: 0.6491
F1: 0.5495

Confusion Matrix (Binary):
               Predicted: Fluent(0)  Predicted: Stutter(1)
True: Fluent(0)      287             189            
True: Stutter(1)     93              172            


{'accuracy': 0.6194331983805668,
 'precision': 0.47645429362880887,
 'recall': 0.6490566037735849,
 'f1': 0.549520766773163,
 'confusion_matrix': array([[287, 189],
        [ 93, 172]])}

## Step 5: Model 2 - Deep Neural Network

In [4]:
deep_nn = DeepNeuralNetwork(
    model_name="Deep_NN_128_64", 
    hidden_layer_sizes=[128, 64], 
    epochs=200,
    lr=0.01,                 # Explicit learning rate
    activation_fn=nn.Sigmoid,    
    optimizer_class=optim.SGD     
)

deep_nn.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
deep_nn.evaluate(X_test_final, y_test)

[Model: Deep_NN_128_64] Initialized.
[Deep_NN_128_64] Training PyTorch Network (Layers: [128, 64]) for 200 epochs...
   Epoch [50/200], Loss: 0.6928
   Epoch [100/200], Loss: 0.6926
   Epoch [150/200], Loss: 0.6924
   Epoch [200/200], Loss: 0.6922

--- Evaluation on Unseen Test Set ---

--- Evaluation: Deep_NN_128_64 ---
Accuracy: 0.5493
Precision: 0.4044
Recall: 0.5509
F1: 0.4665

Confusion Matrix (Binary):
               Predicted: Fluent(0)  Predicted: Stutter(1)
True: Fluent(0)      261             215            
True: Stutter(1)     119             146            


{'accuracy': 0.5492577597840755,
 'precision': 0.40443213296398894,
 'recall': 0.5509433962264151,
 'f1': 0.46645367412140576,
 'confusion_matrix': array([[261, 215],
        [119, 146]])}